# Lighting robustness tests

- **Lighting robustness tests** check whether predictions stay stable under controlled illumination perturbations.
- Perturbations include darkening, brightening, additive offsets, and contrast shifts.
- Claim set for robustness focuses on **non-saturating** conditions (`dark_25`, `bright_25`, offsets, moderate contrast). `dark_50` and `bright_50` are treated as stress conditions and excluded from the claim set.
- Current claim criterion is: **accuracy >= 0.90**, **flip_rate <= 0.07**, and **max absolute recall delta <= 0.02** on claim perturbations.
- The two pipelines differ in input modality: the SVM pipeline uses **grayscale** images for handcrafted texture features, while the transfer model uses full-color **RGB** images.
- This modality difference can matter under illumination changes: RGB inputs may retain chromatic cues that are not available after grayscale conversion.
- Both models use illumination normalization in the preprocessing stage.
- Advantages: it reveals whether model decisions depend on lighting artifacts rather than intrinsic texture features, and it quantifies prediction drift under operating variability.
- Limitations: these tests do not cover all real-world imaging variations (e.g. shadows, reflections, sensor noise), but they do provide a useful first check for illumination sensitivity.

## SVM pipeline
- The configuration uses `illumination_normalization = "percentile"` with `contrast_percentiles = (2.0, 98.0)` to improve robustness under contrast shifts.
- Inputs are converted to grayscale before handcrafted GLCM + DWT extraction, so the model relies primarily on luminance/texture structure.
- Training now includes controlled brightening augmentation so the SVM can better tolerate `bright_25` at inference time.

## Transfer pipeline
- The selected normalization profile is **CLAHE** with conservative strength (`clip_limit = 0.003`) to avoid over-normalization.
- A parameter sweep was used to choose this transfer setting because stronger normalization (for example CLAHE `0.01`) degraded transfer accuracy.
- The transfer model consumes full-color RGB inputs, which may improve robustness to some lighting shifts by preserving channel-specific cues.

## Transfer vs SVM: Key Differences
- **Normalization strategy**:
  - SVM uses **percentile illumination normalization** (`contrast_percentiles = (2.0, 98.0)`) in its classic feature pipeline.
  - Transfer model uses **CLAHE** (`clip_limit = 0.003`) in robustness evaluation to stabilize lighting sensitivity.
- **Input modality**:
  - SVM extracts handcrafted features from grayscale images.
  - Transfer model performs inference on RGB images.
- **Sensitivity pattern**:
  - Transfer model was originally more sensitive to `dark_25` flip-rate and required explicit normalization tuning.

# Algorithmic Fairness
- In this defect inspection scenario, false positives and false negatives have distinct operational consequences.
- A false positive means a good part is flagged as defective, causing unnecessary rework, inspection cost, and lower manufacturing throughput.
- A false negative means a defective part is passed as good, which is more serious because it can lead to downstream failures, safety hazards, and customer dissatisfaction.
- For real-world fairness, the model should prioritize reducing false negatives while keeping false positives low enough to avoid excessive waste and mistrust in automated quality control.
- Confusion matrices, class-specific recall, and robustness checks under varied lighting help reveal whether the system is systematically biased toward one type of error.


In [ ]:
# Shared lighting robustness evaluation for both persisted models:
# 1) classic SVM pipeline (GLCM + DWT)
# 2) transfer-learning CNN model
import importlib
import json
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import tensorflow as tf
from sklearn.pipeline import Pipeline

import modules.steelblast_lighting_analysis as sla
from modules.steelblast_classic_features import FeatureExtractionConfig, load_split

# Reload local module so notebook picks up recent robustness utility edits.
importlib.reload(sla)
build_svm_predict_fn = sla.build_svm_predict_fn
build_transfer_predict_fn = sla.build_transfer_predict_fn
run_robustness_analysis = sla.run_robustness_analysis

# Shared dataset setup
# Uses the same test split for both models so robustness outcomes are directly comparable.
dataset_dir = Path("SteelBlastQC")
test_paths, y_test = load_split(dataset_dir, "test")

# Configurable thresholds for robustness claims
CLAIM = "Model is robust to non-saturating lighting variation."
MIN_CLAIM_ACCURACY = 0.90
MAX_CLAIM_FLIP_RATE = 0.07
MAX_CLAIM_ABS_RECALL_DELTA = 0.02
CRITERION = (
    "For moderate darkening/brightening, additive offsets, and moderate contrast shifts: "
    f"accuracy >= {MIN_CLAIM_ACCURACY:.2f} and "
    f"flip_rate <= {MAX_CLAIM_FLIP_RATE:.2f} and "
    f"max absolute recall delta <= {MAX_CLAIM_ABS_RECALL_DELTA:.2f}."
)
NOTE = "Severe darkening/brightening is reported as stress testing because clipping can destroy texture information."
run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

## SVM Robustness Evaluation

This section evaluates lighting robustness of the persisted SVM model (`steelblast_svm_glcm_dwt.joblib`) on the shared test split used for both models.

### General Approach
- Load the saved SVM pipeline.
- Convert each image to grayscale and extract handcrafted texture descriptors (GLCM + DWT) with percentile-based illumination normalization.
- Generate predictions for each lighting perturbation using the shared robustness utilities.
- Compute per-perturbation metrics: accuracy, prediction flip-rate, class recalls, and confusion matrix.
- Summarize whether the SVM satisfies the lighting robustness claim thresholds.
- Save results to a timestamped JSON artifact for reproducibility.

### Main Parameters
- `svm_model_path`:
  - Path to the persisted SVM pipeline (`steelblast_svm_glcm_dwt.joblib`).
- `svm_feature_config.image_size = 256`:
  - Resolution used before handcrafted feature extraction.
- `input modality = grayscale`:
  - The classic feature pipeline operates on grayscale intensity patterns rather than RGB channels.
- `illumination_normalization = "percentile"`:
  - Contrast normalization strategy for robust texture extraction.
- `contrast_percentiles = (2.0, 98.0)`:
  - Lower/upper clipping bounds used for percentile normalization.
- `glcm_levels = 32`, `glcm_distances = (1, 2, 4, 8)`, `glcm_angles = (0, pi/4, pi/2, 3pi/4)`:
  - Texture co-occurrence settings controlling spatial statistics.
- `glcm_properties = ("contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM")`:
  - Statistical descriptors extracted from GLCM matrices.
- `dwt_wavelet = "db2"`, `dwt_level = 3`:
  - Wavelet decomposition parameters for multi-scale texture features.
- Claim thresholds:
  - `MIN_CLAIM_ACCURACY = 0.90`
  - `MAX_CLAIM_FLIP_RATE = 0.07`
  - `MAX_CLAIM_ABS_RECALL_DELTA = 0.02`


In [ ]:


# Analyze SVM model
svm_model_path = Path("steelblast_svm_glcm_dwt.joblib")
if not svm_model_path.exists():
    raise FileNotFoundError(f"Saved SVM model not found: {svm_model_path.resolve()}")

svm_feature_config = FeatureExtractionConfig(
    image_size=256,
    illumination_normalization="percentile",
    contrast_percentiles=(2.0, 98.0),
    glcm_levels=32,
    glcm_distances=(1, 2, 4, 8),
    glcm_angles=(0.0, np.pi / 4, np.pi / 2, 3 * np.pi / 4),
    glcm_properties=("contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"),
    dwt_wavelet="db2",
    dwt_level=3,
)

svm_model: Pipeline = joblib.load(svm_model_path)
svm_predict_for_perturbation = build_svm_predict_fn(
    image_paths=test_paths,
    feature_config=svm_feature_config,
    fitted_model=svm_model,
)
svm_lighting_robustness = run_robustness_analysis(
    model_name="SVM (GLCM + DWT)",
    y_test=y_test,
    predict_for_perturbation=svm_predict_for_perturbation,
    claim=CLAIM,
    criterion=CRITERION,
    min_accuracy_threshold=MIN_CLAIM_ACCURACY,
    max_flip_rate_threshold=MAX_CLAIM_FLIP_RATE,
    max_abs_recall_delta_threshold=MAX_CLAIM_ABS_RECALL_DELTA,
    note=NOTE,
)

svm_metrics_path = Path(f"bias_analysis_metrics_svm_glcm_dwt_{run_timestamp}.json")
svm_metrics = {
    "dataset_dir": str(dataset_dir),
    "model_key": "svm_glcm_dwt",
    "model_path": str(svm_model_path),
    "lighting_robustness": svm_lighting_robustness,
}
svm_metrics_path.write_text(json.dumps(svm_metrics, indent=2))
print(f"\nSaved SVM lighting robustness results to: {svm_metrics_path.resolve()}")



## Transfer Robustness Evaluation

This section evaluates lighting robustness of the persisted transfer-learning model (`steelblast_resnet50.h5`) on the same test split used for SVM, with explicit illumination normalization enabled in the transfer prediction path.

### General Approach
- Load the saved transfer model.
- Build a transfer prediction function with model-specific preprocessing (`ResNet50 preprocess_input`) on full-color RGB inputs.
- Enable illumination normalization in the transfer path using the selected CLAHE profile.
- Generate predictions for each lighting perturbation through the shared robustness utilities.
- Compute per-perturbation metrics: accuracy, prediction flip-rate, class recalls, and confusion matrix.
- Summarize whether the transfer model satisfies the claim thresholds.
- Save results and normalization settings to a timestamped JSON artifact.

### Main Parameters
- `transfer_model_path`:
  - Path to the persisted transfer model (`steelblast_resnet50.h5`).
- `image_size = (224, 224)`:
  - Input resolution expected by ResNet50.
- `input modality = RGB`:
  - The transfer pipeline keeps 3-channel color information, which may preserve chromatic/reflectance cues under lighting shifts.
- `preprocess_input_fn = tf.keras.applications.resnet50.preprocess_input`:
  - Model-specific pixel normalization before inference.
- `threshold = 0.5`:
  - Probability threshold for binary prediction.
- `use_illumination_normalization = True`:
  - Enables robustness normalization in the transfer prediction path.
- `illumination_normalization = "clahe"`:
  - Selected normalization method for transfer robustness evaluation.
- `clahe_clip_limit = 0.003`:
  - Conservative CLAHE strength selected from parameter sweep.
- `normalize_baseline = True`:
  - Applies the same normalization policy to baseline and perturbed inputs to keep comparisons consistent.
- Claim thresholds:
  - `MIN_CLAIM_ACCURACY = 0.90`
  - `MAX_CLAIM_FLIP_RATE = 0.07`
  - `MAX_CLAIM_ABS_RECALL_DELTA = 0.02`


In [ ]:
# Analyze transfer-learning model
transfer_model_path = Path("steelblast_resnet50.h5")
if not transfer_model_path.exists():
    raise FileNotFoundError(f"Saved transfer model not found: {transfer_model_path.resolve()}")

transfer_model = tf.keras.models.load_model(transfer_model_path, compile=False)
transfer_norm_method = "clahe"
transfer_clahe_clip_limit = 0.003
transfer_predict_for_perturbation = build_transfer_predict_fn(
    image_paths=test_paths,
    model=transfer_model,
    image_size=(224, 224),
    preprocess_input_fn=tf.keras.applications.resnet50.preprocess_input,
    threshold=0.5,
    use_illumination_normalization=True,
    illumination_normalization=transfer_norm_method,
    clahe_clip_limit=transfer_clahe_clip_limit,
    normalize_baseline=True,
)
transfer_lighting_robustness = run_robustness_analysis(
    model_name=f"Transfer Learning (ResNet50 + {transfer_norm_method}, clip={transfer_clahe_clip_limit})",
    y_test=y_test,
    predict_for_perturbation=transfer_predict_for_perturbation,
    claim=CLAIM,
    criterion=CRITERION,
    min_accuracy_threshold=MIN_CLAIM_ACCURACY,
    max_flip_rate_threshold=MAX_CLAIM_FLIP_RATE,
    max_abs_recall_delta_threshold=MAX_CLAIM_ABS_RECALL_DELTA,
    note=NOTE,
)

transfer_metrics_path = Path(f"bias_analysis_metrics_transfer_resnet50_{run_timestamp}.json")
transfer_metrics = {
    "dataset_dir": str(dataset_dir),
    "model_key": "transfer_resnet50",
    "model_path": str(transfer_model_path),
    "lighting_robustness": transfer_lighting_robustness,
    "normalization": {
        "method": transfer_norm_method,
        "clahe_clip_limit": transfer_clahe_clip_limit,
        "normalize_baseline": True,
    },
}
transfer_metrics_path.write_text(json.dumps(transfer_metrics, indent=2))
print(f"Saved transfer lighting robustness results to: {transfer_metrics_path.resolve()}")

# Bias analysis comparison

- Both models pass the stated robustness claim for non-saturating lighting.
- ResNet50 is more accurate across most moderate perturbations.
- SVM is more stable in prediction consistency (lower flip rates) within the claim set.
- Under severe brightening (`bright_50` stress test), ResNet50 is clearly stronger.

| Metric | SVM GLCM+DWT | Transfer ResNet50 | Better |
|----------|------------:|------------------:|---------|
| Claim passed | true | true | Tie |
| Min claim accuracy | 0.932 | 0.932 | Tie |
| Max claim flip rate | 0.020 | 0.052 | SVM (more stable) |
| Max claim abs recall delta | 0.014 | 0.018 | SVM (more stable) |
| Avg claim-set accuracy | 0.935 | 0.943 | ResNet50 |
| Avg claim-set flip rate | 0.007 | 0.020 | SVM |

## Per-perturbation highlights

- `dark_25`, `offset_plus_10`, `offset_minus_10`, `low_contrast`, `high_contrast`: ResNet50 has higher accuracy.
- `bright_25`: SVM is better (0.940 vs 0.932) and flips less (0.020 vs 0.052).
- `bright_50` (stress only): ResNet50 is much better (accuracy 0.904 vs 0.792; flip rate 0.088 vs 0.200).

## Class-wise behavior

- SVM tends to preserve higher `not_good` recall under lighting shifts.
- ResNet50 tends to preserve higher `good` recall and higher overall accuracy.
